# Evals en el SDLC con IA

## Cómo dejar de adivinar y empezar a medir

---

**Anthropic Partner Basecamp**

*Claude Evals Workshop — Lab 00: Introducción*

# El problema: "vibe coding"

- Pides a Claude que escriba una función → parece funcionar ✓
- Cambias el prompt ligeramente → la salida cambia sin avisar
- En producción, nadie sabe si el comportamiento es **correcto** o simplemente *plausible*

---

> "Funciona en mis pruebas manuales"
> — Todo el mundo, antes del incidente en producción

**Sin medición sistemática:** ojo humano → intuición → deploy

# Qué es un evaluador (eval)

Un **evaluador** es una función que toma la salida de un LLM y devuelve una señal de calidad medible.

```
evaluate(output, expected) → score | pass/fail
```

| Testing tradicional | Evaluador de LLM |
|---------------------|-----------------|
| `assert result == expected` | `score = evaluate(llm_output)` |
| Determinista | Puede ser probabilístico |
| Falla o pasa | Gradiente 0.0 – 1.0 |
| CI lo ejecuta automáticamente | CI lo ejecuta automáticamente |

**Los evaluadores son tests para comportamiento no determinista.**

# Tipo 1: Exact Match

## Comparación literal, 100% determinista

```python
def exact_match(output: str, expected: str) -> bool:
    return output.strip() == expected.strip()
```

> **Cuándo usarlo:** extracción de campos estructurados, clasificación con etiquetas fijas, respuestas de una sola palabra.

## Exact Match — Limitaciones

- No tolera variación legítima de lenguaje natural
- Frágil ante cambios de formato triviales (mayúsculas, espacios)

---

> **Regla de oro:** si la respuesta correcta tiene exactamente una forma posible, usa Exact Match.

# Tipo 2: Rule-Based

## Lógica Python, sin LLM

```python
def rule_based_check(output: str) -> float:
    score = 0.0
    if len(output) > 50:                    score += 0.25
    if "```" in output:                     score += 0.25
    if output.count("\n") >= 3:            score += 0.25
    if not output.startswith("Lo siento"):  score += 0.25
    return score
```

> **Cuándo usarlo:** verificar presencia de secciones obligatorias, restricciones de longitud, cualquier regla expresable en Python.

## Rule-Based — Ventajas

- ✅ Rápido, barato, reproducible
- ✅ Sin dependencia de red ni coste de tokens
- ✅ Resultado determinista — ideal para CI/CD

---

> Componer varios checks en una puntuación agregada (0.0 – 1.0).

# Tipo 3: LLM-as-Judge

## Claude evalúa la calidad

```python
judge_prompt = f"""
Evalúa la respuesta. Puntúa de 0 a 10 en: corrección,
claridad y seguridad. Solo JSON: {{"score": X, "razon": "..."}}

Respuesta: {output}
"""
result = claude(judge_prompt)
```

> **Cuándo usarlo:** calidad subjetiva, tono, claridad, cuando "correcto" tiene múltiples formas válidas.

## LLM-as-Judge — Consideraciones

- ⚠️ Más lento y costoso que los otros tipos
- ⚠️ Requiere meta-evaluación: ¿el juez es fiable?
- ✅ Cubre casos imposibles de expresar con lógica Python

---

> Diseña y testea el prompt del juez con el mismo rigor que el prompt principal.

# El ciclo evaluación-driven

```
┌─────────────────────────────────────────────────┐
│  BASELINE       CAMBIO            MEDIR         │
│                                                 │
│  Mide calidad → Modifica       → Mide calidad  │
│  en v_actual    prompt/modelo    en v_nueva     │
│                      ↑                          │
│                   ITERAR                        │
│             (si score no mejora)                │
└─────────────────────────────────────────────────┘
```

> **No merges sin métricas.**
> Si no tienes un criterio medible de "listo", no tienes un criterio de "listo".

# Los 4 labs del workshop

| Lab | Título | Qué aprendes |
|-----|--------|--------------|
| **01** | SDLC Gatekeeper | Evaluadores como gate en CI/CD |
| **02** | Architect Agent | Medir razonamiento de un agente |
| **03** | Performance | TTFT, TTC, tokens y coste por modelo |
| **04** | Pipeline E2E | Combinar los 3 labs en CI/CD |

---

Cada lab es **autocontenido** — puedes ejecutarlo de forma independiente.

# Preguntas

---

## Recursos

- **Docs Anthropic**: [docs.anthropic.com/en/docs/test-evaluate-prompts](https://docs.anthropic.com/en/docs/test-evaluate-prompts)
- **Este workshop**: carpeta `labs/` en el repositorio
- **Spec de diseño**: `docs/superpowers/specs/2026-05-20-evals-workshop-design.md`

---

*Claude Evals Workshop — Anthropic Partner Basecamp*